Prediksi apakah seorang karyawan akan resign (1) atau bertahan (0)*berdasarkan:
- Kepuasan kerja (skala 1–10)
- Beban kerja (jam/minggu)
- Gaji (juta rupiah/bulan)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data_karyawan = {
    'kepuasan_kerja': [8, 3, 7, 2, 9, 4, 6, 1, 7, 5, 3, 8, 2, 9, 4],
    'beban_kerja_jam': [40, 60, 42, 70, 38, 65, 45, 75, 40, 55, 68, 38, 72, 35, 62],
    'gaji_juta':       [8,  4,  7,  3,  10, 4,  6,  2,  9,  5,  3,  9,  3,  11, 4 ],
    'resign':          [0,  1,  0,  1,  0,  1,  0,  1,  0,  1,  1,  0,  1,  0,  1 ]
}

df_karyawan = pd.DataFrame(data_karyawan)
print("Dataset Karyawan:")
print(df_karyawan.to_string(index=False))
print(f"\nDistribusi label: Bertahan={sum(df_karyawan['resign']==0)}, Resign={sum(df_karyawan['resign']==1)}")

In [ ]:
X_kar = df_karyawan[['kepuasan_kerja', 'beban_kerja_jam', 'gaji_juta']]
y_kar = df_karyawan['resign']

X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X_kar, y_kar, test_size=0.2, random_state=3
)

model_logistik = LogisticRegression(max_iter=200)
model_logistik.fit(X_train_k, y_train_k)

print("Intercept:", model_logistik.intercept_)
print("Koefisien [kepuasan, beban_kerja, gaji]:", model_logistik.coef_)

In [ ]:
y_pred_logistik = model_logistik.predict(X_test_k)
y_prob = model_logistik.predict_proba(X_test_k)[:, 1]

print("Hasil Prediksi:")
hasil_k = pd.DataFrame({
    'Aktual'   : y_test_k.values,
    'Prediksi' : y_pred_logistik,
    'Prob Resign': y_prob.round(2)
})
print(hasil_k.to_string(index=False))

print(f"\nAkurasi: {accuracy_score(y_test_k, y_pred_logistik):.2f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_k, y_pred_logistik))
print("\nClassification Report:")
print(classification_report(y_test_k, y_pred_logistik, target_names=['Bertahan','Resign']))

In [ ]:
karyawan_baru = [[3, 65, 3.5]]  # kepuasan rendah, beban tinggi, gaji kecil
prediksi_resign = model_logistik.predict(karyawan_baru)
prob_resign = model_logistik.predict_proba(karyawan_baru)[0][1]
status = 'RESIGN' if prediksi_resign[0] == 1 else 'BERTAHAN'
print(f"Prediksi karyawan baru: {status} (probabilitas resign: {prob_resign:.1%})")

import seaborn as sns
cm = confusion_matrix(y_test_k, y_pred_logistik)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Bertahan','Resign'],
            yticklabels=['Bertahan','Resign'])
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix — Regresi Logistik')
plt.tight_layout()
plt.show()

**ANALISIS :**
- **Koefisien kepuasan_kerja** bernilai negatif terhadap resign → karyawan yang puas cenderung tidak resign.
- **Koefisien beban_kerja** bernilai positif → semakin tinggi beban kerja, probabilitas resign meningkat.
- **Koefisien gaji** bernilai negatif → gaji lebih tinggi menurunkan kemungkinan resign.
- Berbeda dari regresi linear, output regresi logistik adalah **probabilitas** (0–1) yang kemudian dikonversi ke kelas 0 atau 1 menggunakan threshold 0.5.
- Model ini berguna untuk HR dalam mengidentifikasi karyawan yang berisiko tinggi resign agar dapat dilakukan intervensi lebih awal.